# NHL expected goals (xG) model

Reads the shot CSVs produced by the scraper notebook (`shots_*.csv` on Google Drive) and builds the xG model.

**The CSVs must come from the latest scraper notebook (`nhl_xg_scraper.ipynb`).** Older CSVs will fail to load with a message saying so.

**Run the cells top to bottom** (Runtime → Run all). Sections 1–7 are quick. Sections 8–14 each fit many models and are marked with their rough run time; a full run takes about 3–4 hours. Sections 14–18 produce the final xG values, the diagnostics and the rink chart. Section 19 (what drives the model) can also be run on its own after sections 1–5.

| Section | What it does |
|---|---|
| 1–2 | Setup, load and filter shots |
| 3 | Arena adjustment (off by default) |
| 4 | Features: shot, pre-shot context, shift fatigue, handedness, sequence |
| 5–7 | Season split, baseline, single model, per-state models |
| 8 | Hyperparameter tuning |
| 9 | Test: do the sequence features and the tuning really help? |
| 10 | Season half-life search |
| 11–13 | Holdout check, walk-forward backtest, in-season calibration test |
| 14–16 | Backfill xG for every shot, danger-bucket check, calibration fix |
| 17 | Flurry adjustment and saving the results |
| 18 | xG by rink location chart |
| 19 | What drives the model: SHAP beeswarm |

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import glob, gc
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from scipy.optimize import brentq
from scipy.special import logit, expit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.calibration import calibration_curve

DATA_DIR = '/content/drive/MyDrive'   # folder holding shots_*.csv
H = 1                                 # season half-life in years (chosen in section 10)
ARENA_ADJUST = False                  # rink recording-bias correction (section 3); testing showed it hurts
N_TRIALS = 30                         # hyperparameter settings tried per strength state (section 8)
USE_TUNED = True                      # use the tuned settings from section 8 in everything after it
FLURRY_GAP = 3                        # seconds between a team's shots that still count as one flurry (section 17)
TRAIN_END, VALID, TEST = 2023, 2024, 2025

## 2. Load and filter shots

Keeps only the columns the model needs and filters each file as it loads, to stay well inside Colab's memory.

Dropped: blocked shots, short misses and failed bank attempts (only recorded from 2023-24 on), penalty shots, and non-empty-net shots from beyond center ice. `miss_reason` is used for filtering only and must never be a model feature, because it describes the outcome of the shot.

In [ ]:
SHIFT = ['shooter_toi', 'own_avg_toi', 'opp_min_toi', 'opp_avg_toi', 'opp_max_toi']
cols = ['game_id', 'event_id', 'event', 'is_goal', 'team', 'is_home', 'home_team', 'x', 'y',
        'distance', 'angle', 'shot_type', 'miss_reason', 'prev_event', 'prev_team_same',
        'prev_short_miss', 'sec_since_prev', 'dist_from_prev', 'rebound', 'strength',
        'empty_net_against', 'score_diff', 'period', 'game_seconds', 'shooter_pos', 'shooter_hand',
        'prev_x', 'prev_y', 'sec_since_faceoff', 'sec_since_state_change',
        'prev2_event', 'prev2_team_same', 'sec_since_prev2', 'prev2_x', 'prev2_y',
        'sec_in_zone', 'team_shots_10s', 'team_shots_30s'] + SHIFT

def load(f):
    try:
        d = pd.read_csv(f, usecols=cols)
    except ValueError as e:
        raise ValueError(f"{f} was made by an older scraper. Delete it, or re-run the scraper "
                         f"notebook (nhl_xg_scraper.ipynb). ({e})") from None
    d = d[(d.event != 'blocked-shot') & d.x.notna() &
          ~d.miss_reason.isin(['short', 'failed-bank-attempt']) &
          ~d.strength.isin(['1v0', '0v1']) &                      # penalty shots
          ((d.x > -25) | (d.empty_net_against == 1))]
    for c in d.select_dtypes('float64').columns:
        d[c] = d[c].astype('float32')
    return d

files = sorted(glob.glob(f'{DATA_DIR}/shots_*.csv'))
assert files, f"No shots_*.csv files found in {DATA_DIR}. Is Drive mounted?"

df = pd.concat([load(f) for f in files], ignore_index=True)
df = df.drop_duplicates(['game_id', 'event_id']).reset_index(drop=True)
df['season'] = (df.game_id // 1_000_000).astype('int16')     # 2024020001 -> 2024
df['away_team'] = df.game_id.map(df[df.is_home == 0].groupby('game_id').team.first())
gc.collect()

t = df.groupby('season').agg(shots=('is_goal', 'size'), goals=('is_goal', 'sum'))
t['goal_rate'] = (t.goals / t.shots).round(4)
print(t)

## 3. Arena adjustment

Scorekeepers in different rinks record shot locations slightly differently. For each team and season, this compares shots in that team's home games (both teams) with shots in its road games (both teams), and remaps the home-rink coordinates so the two distributions match (quantile matching, after Schuckers and Curro). `x0`, `y0`, `distance0` and `angle0` keep the unadjusted values.

**This is off by default.** In a game-grouped test over all seasons it made the model slightly but measurably worse, alone and combined with the other features. Set `ARENA_ADJUST = True` in section 1 to try it. The table shows the rink-seasons with the largest gap in average shot distance between home and road games, before and after.

In [ ]:
def qmap(vals, src, ref):
    """Move `vals` from the `src` distribution onto the `ref` distribution (mid-rank quantiles)."""
    s = np.sort(src)
    q = (np.searchsorted(s, vals, 'left') + np.searchsorted(s, vals, 'right')) / (2 * len(s))
    return np.quantile(ref, q)

for c in ['x', 'y', 'distance', 'angle']:
    df[c + '0'] = df[c]

if ARENA_ADJUST:
    x_adj, y_adj = df.x.to_numpy('float64').copy(), df.y.abs().to_numpy('float64').copy()
    for (S, T), home in df.groupby(['season', 'home_team'], observed=True):
        road = df[(df.season == S) & (df.away_team == T)]
        if len(home) < 500 or len(road) < 500:
            continue
        x_adj[home.index] = qmap(home.x.values, home.x.values, road.x.values)
        y_adj[home.index] = qmap(home.y.abs().values, home.y.abs().values, road.y.abs().values)
    df['x'] = x_adj.astype('float32')
    df['y'] = (np.sign(df.y0) * y_adj).astype('float32')
    df['distance'] = np.hypot(89 - df.x, df.y).astype('float32')
    df['angle'] = np.degrees(np.arctan2(df.y.abs(), 89 - df.x)).astype('float32')

chk  = df[df.empty_net_against != 1]
home = chk.groupby(['season', 'home_team'], observed=True)[['distance0', 'distance']].mean()
road = chk.groupby(['season', 'away_team'], observed=True).distance0.mean()
home.index.names = road.index.names = ['season', 'rink']
bias = pd.DataFrame({'before': home.distance0 - road, 'after': home.distance - road}).dropna().round(2)
print("Average shot distance, home games minus road games (feet):")
print(bias.reindex(bias.before.abs().sort_values(ascending=False).index).head(10))
print(f"\ntypical gap across all rink-seasons: before {bias.before.abs().mean():.2f} ft, "
      f"after {bias.after.abs().mean():.2f} ft")

## 4. Features

Five groups:

- **Shot and game state** (the original set).
- **Pre-shot context:** where the previous event happened relative to the net, how far and how fast the puck moved, the change in shooting angle, a rush flag (previous event outside the attacking zone within 4 seconds), and time since the last faceoff and since the strength state changed.
- **Shift fatigue:** seconds into the current shift for the shooter, his teammates (average) and the defending skaters (min, average, max).
- **Handedness:** shooting hand, lateral position relative to it, and an off-wing flag.
- **Sequence:** time since play was last outside the attacking zone, the shooting team's shot attempts in the last 10 and 30 seconds, and the event before the previous one (type, team, timing, location). Tested in section 9.

In [ ]:
own, opp = df.strength.str.split('v', expand=True).astype(float).T.values
df['state'] = np.select([df.empty_net_against == 1, own > opp, own < opp], ['EN', 'PP', 'SH'], 'EV')

# --- shot and game state
df['sec_since_prev'] = df.sec_since_prev.clip(0, 60)
df['speed'] = df.dist_from_prev / df.sec_since_prev.clip(lower=1)
df['behind_net'] = (df.x > 89).astype(int)
df['score_diff'] = df.score_diff.clip(-3, 3)
df['is_D'] = (df.shooter_pos == 'D').astype(int)

# --- pre-shot context (unadjusted coordinates, so the shot and the previous event share a frame)
shot_ang = np.degrees(np.arctan2(df.y0, 89 - df.x0))
prev_ang = np.degrees(np.arctan2(df.prev_y, 89 - df.prev_x))
df['prev_dist_net'] = np.hypot(89 - df.prev_x, df.prev_y)
df['dist_change']   = df.prev_dist_net - df.distance0            # > 0: puck moved toward the net
df['angle_change']  = (shot_ang - prev_ang).abs()
df['angle_speed']   = df.angle_change / df.sec_since_prev.clip(lower=1)
df['rush']       = ((df.prev_x < 25) & (df.sec_since_prev <= 4)).astype(int)
df['royal_road'] = ((np.sign(df.y0) * np.sign(df.prev_y) == -1) & (df.sec_since_prev <= 4)).astype(int)
df['sec_since_faceoff'] = df.sec_since_faceoff.clip(0, 120)
df['sec_since_state_change'] = df.sec_since_state_change.clip(0, 180)

# --- sequence: sustained pressure and the event before the previous one
df['sec_in_zone'] = df.sec_in_zone.clip(0, 120)
df['sec_since_prev2'] = df.sec_since_prev2.clip(0, 90)
df['prev2_dist_net'] = np.hypot(89 - df.prev2_x, df.prev2_y)

# --- shift fatigue
for c in SHIFT:
    df[c] = df[c].clip(0, 180)

# --- handedness and off-wing
hand = df.shooter_hand.map({'L': 1.0, 'R': -1.0}).astype('float32')
df['hand_L'] = (hand == 1).where(hand.notna()).astype('float32')
df['y_hand'] = df.y * hand                                      # > 0: shooting from his strong side
pts = (df.shooter_pos == 'D') & (df.distance > 40) & hand.notna()
strong = (df.y_hand[pts] > 0).mean()
if strong < 0.5:                                                # y axis runs the other way: flip
    df['y_hand'] = -df.y_hand
df['off_wing'] = (df.y_hand < 0).where(df.y_hand.notna()).astype('float32')
print(f"defensemen point shots from their strong side: {max(strong, 1 - strong):.0%} "
      f"(most defensemen play their strong side, so this should be well above 50%)")

for c in ['shot_type', 'prev_event', 'prev2_event', 'strength', 'state']:
    df[c] = df[c].fillna('none').astype('category')

F_BASE = ['distance', 'angle', 'x', 'y', 'shot_type', 'prev_event', 'prev_team_same',
          'prev_short_miss', 'sec_since_prev', 'dist_from_prev', 'speed', 'rebound',
          'royal_road', 'behind_net', 'strength', 'state', 'empty_net_against',
          'score_diff', 'is_home', 'period', 'game_seconds', 'is_D']
F_PRESHOT = ['prev_x', 'prev_dist_net', 'dist_change', 'angle_change', 'angle_speed', 'rush',
             'sec_since_faceoff', 'sec_since_state_change']
F_SHIFT = SHIFT
F_HAND  = ['hand_L', 'y_hand', 'off_wing']
F_SEQ   = ['sec_in_zone', 'team_shots_10s', 'team_shots_30s', 'prev2_event', 'prev2_team_same',
           'sec_since_prev2', 'prev2_x', 'prev2_dist_net']
features = F_BASE + F_PRESHOT + F_SHIFT + F_HAND + F_SEQ
states = ['EV', 'PP', 'SH', 'EN']

cov = df.assign(shifts=df.shooter_toi.notna(), hand=df.shooter_hand.notna(), prev_loc=df.prev_x.notna(),
                zone_time=df.sec_in_zone.notna())
print("\nshare of shots with each kind of data:")
print(cov.groupby('season')[['shifts', 'hand', 'prev_loc', 'zone_time']].mean().round(3))
print("\n", df.state.value_counts().to_dict())

## 5. Season split and shared helpers

Splits are by season, never by random row. Train on the past, validate on the next season, hold out the one after.

In [ ]:
train = df[df.season <= TRAIN_END]
valid = df[df.season == VALID]
test  = df[df.season == TEST]

params = dict(n_estimators=3000, learning_rate=0.03, enable_categorical=True, tree_method='hist',
              eval_metric='logloss')
# Per-state model settings. These are the starting values; section 8 searches for better ones.
hp_default = {'EV': dict(max_depth=5, min_child_weight=10, subsample=0.8, colsample_bytree=0.8),
              'PP': dict(max_depth=4, min_child_weight=20, subsample=0.8, colsample_bytree=0.8),
              'SH': dict(max_depth=3, min_child_weight=30, subsample=0.8, colsample_bytree=0.8),
              'EN': dict(max_depth=3, min_child_weight=20, subsample=0.8, colsample_bytree=0.8)}
hp = hp_default

def new_model(s, hp_set=None, **over):
    return xgb.XGBClassifier(**(params | (hp_set or hp)[s] | over))

def fit_states(train, valid, h=None, feats=None, hp_set=None):
    """One model per strength state, early-stopped on `valid`.
    h = season half-life in years (None = all seasons weighted equally)."""
    feats = feats or features
    pred, models = np.full(len(valid), np.nan), {}
    for s in states:
        tr_m, va_m = (train.state == s).values, (valid.state == s).values
        w = None if h is None else 0.5 ** ((valid.season.max() - train.loc[tr_m, 'season']) / h)
        m = new_model(s, hp_set, early_stopping_rounds=100)
        m.fit(train.loc[tr_m, feats], train.is_goal[tr_m], sample_weight=w,
              eval_set=[(valid.loc[va_m, feats], valid.is_goal[va_m])], verbose=False)
        models[s] = m
        pred[va_m] = m.predict_proba(valid.loc[va_m, feats])[:, 1]
    return models, pred

def tree_counts(hp_set=None, feats=None):
    """Trees per state, from early stopping on the validation season with half-life H."""
    models, _ = fit_states(train, valid, h=H, feats=feats, hp_set=hp_set)
    return {s: models[s].best_iteration + 1 for s in states}

def fit_fixed(tr, te, ntrees, target_season, feats=None):
    """Per-state models with fixed tree counts and half-life H weights. Returns predictions for `te`."""
    feats = feats or features
    pred = np.full(len(te), np.nan)
    for s in states:
        tm, em = (tr.state == s).values, (te.state == s).values
        w = 0.5 ** ((target_season - tr.loc[tm, 'season']) / H)
        m = new_model(s, n_estimators=ntrees[s])
        m.fit(tr.loc[tm, feats], tr.is_goal[tm], sample_weight=w)
        pred[em] = m.predict_proba(te.loc[em, feats])[:, 1]
    return pred

def cv_pred(feats, ntrees, hp_set=None):
    """xG for every shot from five game-grouped folds, so no shot is scored by a model that saw its game."""
    p = pd.Series(np.nan, index=df.index)
    for s in states:
        d = df[df.state == s]
        for tr_i, te_i in GroupKFold(n_splits=5).split(d, groups=d.game_id):
            m = new_model(s, hp_set, n_estimators=ntrees[s])
            m.fit(d.iloc[tr_i][feats], d.is_goal.iloc[tr_i])
            p.loc[d.index[te_i]] = m.predict_proba(d.iloc[te_i][feats])[:, 1]
    return p

def report(d, pred, label):
    """Log loss, AUC and xG/goals ratio overall and by state."""
    v = d.assign(p=pred)
    t = v.groupby('state', observed=True).agg(goals=('is_goal', 'sum'), xG=('p', 'sum'))
    print(f"{label}: logloss {log_loss(v.is_goal, v.p):.4f}  AUC {roc_auc_score(v.is_goal, v.p):.3f}  "
          f"overall {v.p.sum() / v.is_goal.sum():.3f}")
    print('   ', (t.xG / t.goals).round(3).to_dict())

def danger(p):
    return pd.cut(p, [0, .05, .15, 1], labels=['low', 'medium', 'high'], include_lowest=True)

def bucket_table(d, col, by):
    """Calibration by group. ratio = xG/goals. z beyond +/-2 is more than noise
    (z > 0: model too low, z < 0: model too high)."""
    t = d.assign(_v=d[col] * (1 - d[col])).groupby(by, observed=True).agg(
        shots=(col, 'size'), goals=('is_goal', 'sum'), xG=(col, 'sum'), _v=('_v', 'sum'))
    t['ratio'] = (t.xG / t.goals).round(3)
    t['z'] = ((t.goals - t.xG) / np.sqrt(t._v)).round(2)
    return t.drop(columns='_v')

## 6. Baseline and single XGBoost model

Reference points: a distance-and-angle logistic regression, then one XGBoost model on all features.

In [ ]:
b = LogisticRegression(max_iter=1000).fit(train[['distance', 'angle']], train.is_goal)
p = b.predict_proba(valid[['distance', 'angle']])[:, 1]
base_ll = log_loss(valid.is_goal, p)
print(f"baseline  logloss {base_ll:.4f}  AUC {roc_auc_score(valid.is_goal, p):.3f}")

single = xgb.XGBClassifier(**(params | hp_default['EV']), early_stopping_rounds=100)
single.fit(train[features], train.is_goal, eval_set=[(valid[features], valid.is_goal)], verbose=200)
valid = valid.assign(xg=single.predict_proba(valid[features])[:, 1])
report(valid, valid.xg, "single model")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4))

curve = single.evals_result()['validation_0']['logloss']
a1.plot(curve, label=f'XGBoost ({VALID} validation)')
a1.axvline(single.best_iteration, color='gray', ls=':', label=f'best iteration ({single.best_iteration})')
a1.axhline(base_ll, color='orange', ls='--', label='distance + angle baseline')
a1.set_xlabel('trees'); a1.set_ylabel('log loss'); a1.set_ylim(min(curve) - 0.005, base_ll + 0.015)
a1.set_title('Validation curve'); a1.legend()

frac, mean = calibration_curve(valid.is_goal, valid.xg, n_bins=20, strategy='quantile')
a2.plot(mean, frac, 'o-'); a2.plot([0, .5], [0, .5], '--')
a2.set_xlabel('predicted'); a2.set_ylabel('actual'); a2.set_title('Calibration (single model)')
plt.show()

## 7. Per-state models

Separate models for even strength, power play, shorthanded and empty net. *(about 2–5 min)*

In [ ]:
_, pred = fit_states(train, valid)
report(valid, valid.xg, "single model")
report(valid, pred,     "per-state   ")

## 8. Hyperparameter tuning

Random search over the model settings (tree depth, minimum leaf weight, row and column sampling, regularization), separately for each strength state. Each setting is scored by average log loss over two walk-forward folds: train on the seasons before the validation season and before the one preceding it, validate on each. The test season is never used. The starting settings are always included, so the result can only match or beat them on this score.

`N_TRIALS` in section 1 controls how many settings are tried per state. Section 9 then checks whether the tuned settings really are better. *(about 45–90 min at 30 trials)*

In [ ]:
rng = np.random.default_rng(0)

def sample_settings():
    return dict(max_depth=int(rng.integers(3, 9)),
                min_child_weight=round(float(np.exp(rng.uniform(np.log(5), np.log(150)))), 1),
                subsample=round(float(rng.uniform(0.6, 1.0)), 2),
                colsample_bytree=round(float(rng.uniform(0.5, 1.0)), 2),
                reg_lambda=round(float(np.exp(rng.uniform(np.log(0.5), np.log(20)))), 2),
                gamma=round(float(rng.uniform(0, 2)), 2))

def search_score(s, cfg):
    """Average validation log loss over two walk-forward folds (faster learning rate for the search)."""
    total = 0
    for V in (VALID - 1, VALID):
        tr = df[(df.season < V) & (df.state == s)]
        va = df[(df.season == V) & (df.state == s)]
        m = xgb.XGBClassifier(**(params | cfg | dict(learning_rate=0.1, early_stopping_rounds=40)))
        m.fit(tr[features], tr.is_goal, sample_weight=0.5 ** ((V - tr.season) / H),
              eval_set=[(va[features], va.is_goal)], verbose=False)
        total += m.best_score
    return total / 2

hp_tuned = {}
for s in states:
    trials = [hp_default[s]] + [sample_settings() for _ in range(N_TRIALS)]
    scores = [search_score(s, cfg) for cfg in trials]
    best = int(np.argmin(scores))
    hp_tuned[s] = trials[best]
    print(f"{s}: starting {scores[0]:.5f} -> best {scores[best]:.5f}  {trials[best]}")

hp = hp_tuned if USE_TUNED else hp_default
ntrees = tree_counts()
print("\nusing", "tuned" if USE_TUNED else "starting", "settings; trees per state:", ntrees)

## 9. Do the sequence features and the tuning really help?

Scores every shot from all seasons with five game-grouped folds under each setup, then compares them shot by shot. This is far more reliable than one validation season. **A negative difference with z below about −3 is a real improvement; z between −2 and +2 means no real difference.**

If the sequence features do not clear that bar, remove `F_SEQ` from `features` in section 4. If the tuning does not, set `USE_TUNED = False` in section 1. *(about 45–75 min; safe to skip)*

In [ ]:
df['season_cat'] = df.season.astype('category')
prev_build = F_BASE + F_PRESHOT + F_SHIFT + F_HAND
nt_default = tree_counts(hp_default)

def loss(p):
    return -(df.is_goal * np.log(p) + (1 - df.is_goal) * np.log(1 - p))

L = {'original features':         loss(cv_pred(F_BASE + ['season_cat'], ntrees)),
     'previous build':            loss(cv_pred(prev_build + ['season_cat'], ntrees)),
     'all features':              loss(cv_pred(features + ['season_cat'], ntrees)),
     'all features, not tuned':   loss(cv_pred(features + ['season_cat'], nt_default, hp_default))}

def compare(a, b, label):
    g = (L[a] - L[b]).groupby(df.game_id).sum()      # per-game difference
    print(f"{label:38s} {L[a].mean() - L[b].mean():+.5f}   z = {g.mean() / (g.std() / np.sqrt(len(g))):+.1f}")

for k, v in L.items():
    print(f"{k:26s} logloss {v.mean():.4f}")
print()
compare('previous build', 'original features', 'previous build vs original')
compare('all features', 'previous build', 'sequence features (all vs previous)')
compare('all features', 'all features, not tuned', 'tuning (tuned vs not tuned)')

## 10. Season half-life search

Weights each training season by `0.5 ** (years back / h)`. Pick the `h` whose state ratios sit closest to 1.00 without losing AUC, and set `H` in section 1 to match (currently 1). *(about 15–30 min)*

In [ ]:
hl = {}
for h in [1, 2, 3, 5]:
    hl[h] = fit_states(train, valid, h=h)
    report(valid, hl[h][1], f"half-life {h}")

ntrees = {s: hl[H][0][s].best_iteration + 1 for s in states}
print("trees per state at H =", H, ntrees)

## 11. Holdout check on the test season

Retrains through the validation season with the tree counts above, then scores the test season. This season has already been looked at during development, so treat it as a check, not a clean test. *(about 2 min)*

In [ ]:
test_pred = fit_fixed(df[df.season <= VALID], test, ntrees, TEST)
report(test, test_pred, f"TEST {TEST}")

## 12. Walk-forward backtest

Each season is scored by models trained only on earlier seasons. Shows how far off the overall level is when scoring a season the model has not seen. *(about 10–20 min)*

In [ ]:
rows = []
for S in range(df.season.min() + 1, df.season.max() + 1):
    te = df[df.season == S]
    te = te.assign(p=fit_fixed(df[df.season < S], te, ntrees, S))
    r = {'season': S, 'logloss': log_loss(te.is_goal, te.p), 'auc': roc_auc_score(te.is_goal, te.p),
         'overall': te.p.sum() / te.is_goal.sum()}
    r |= {s: te[te.state == s].p.sum() / te[te.state == s].is_goal.sum() for s in states}
    rows.append(r); print(S, 'done')

wf = pd.DataFrame(rows).round(3)
wf

## 13. In-season calibration test

Simulates live use on the test season in six blocks of games. Each block is scored by a model that has seen the season's earlier games, then a season-to-date shift is fit on the earlier blocks only and applied. The shift is what the live model should use. *(about 15–30 min)*

In [ ]:
S = TEST
games  = np.sort(df.loc[df.season == S, 'game_id'].unique())
blocks = np.array_split(games, 6)
prior  = df[df.season < S]

out = []
for k, blk in enumerate(blocks):
    seen = df[(df.season == S) & (df.game_id < blk[0])]
    te = df[df.game_id.isin(blk)]
    out.append(te.assign(p=fit_fixed(pd.concat([prior, seen]), te, ntrees, S)))
    print(f"block {k} scored ({seen.game_id.nunique()} games seen)")
res = pd.concat(out)

adj = []
for k, blk in enumerate(blocks):
    cur, past = res[res.game_id.isin(blk)], res[res.game_id < blk[0]]
    shift = 0.0 if past.empty else brentq(
        lambda b: expit(logit(past.p) + b).sum() - past.is_goal.sum(), -1, 1)
    cur = cur.assign(p_adj=expit(logit(cur.p) + shift))
    adj.append(cur)
    print(f"block {k}: shift {shift:+.3f}  raw {cur.p.sum() / cur.is_goal.sum():.3f}  "
          f"adjusted {cur.p_adj.sum() / cur.is_goal.sum():.3f}")
adj = pd.concat(adj)
print(f"\nseason: raw {adj.p.sum() / adj.is_goal.sum():.3f}  adjusted {adj.p_adj.sum() / adj.is_goal.sum():.3f}")

## 14. Backfill: xG for every shot, 2019 onward

Five folds split by game, so no shot is scored by a model that saw its own game. Season is a feature here, which lets the model learn each season's level. Do not use the season feature in a live model. *(about 10–20 min)*

In [ ]:
df['season_cat'] = df.season.astype('category')
df['xg'] = cv_pred(features + ['season_cat'], ntrees)

print(bucket_table(df, 'xg', 'season')[['goals', 'xG', 'ratio']])
print(f"logloss {log_loss(df.is_goal, df.xg):.4f}  AUC {roc_auc_score(df.is_goal, df.xg):.3f}")

## 15. Calibration by danger level

Low = xG under 5%, medium = 5–15%, high = over 15%. A bucket is worth fixing only if `z` is beyond ±2 and keeps the same sign across seasons.

In [ ]:
df['danger'] = danger(df.xg)
print(bucket_table(df, 'xg', 'danger'), '\n')
print(bucket_table(df, 'xg', ['state', 'danger']), '\n')
sd = bucket_table(df, 'xg', ['season', 'danger'])
print(sd['ratio'].unstack(), '\n')
print(sd['z'].unstack())

## 16. Calibration fix and final xG

In the first build, even-strength shots from 2023 on were too compressed (weak shots rated too high, dangerous shots too low). Check section 15 to see whether that still holds with the new features; a slope `b` close to 1.00 below means the fix is doing almost nothing. This fits `logit(p') = a + b * logit(p)` on those shots with game-grouped folds to produce `xg_final`.

In [ ]:
df['xg_final'] = df.xg
mask = ((df.state == 'EV') & (df.season >= 2023)).values
d = df[mask]
X = logit(d.xg.clip(1e-4, 1 - 1e-4)).to_frame('lx')

fixed = np.empty(len(d))
for tr_i, te_i in GroupKFold(n_splits=5).split(d, groups=d.game_id):
    lr = LogisticRegression(C=1e6).fit(X.iloc[tr_i], d.is_goal.iloc[tr_i])
    fixed[te_i] = lr.predict_proba(X.iloc[te_i])[:, 1]
df.loc[mask, 'xg_final'] = fixed

lr_all = LogisticRegression(C=1e6).fit(X, d.is_goal)
print(f"slope b = {lr_all.coef_[0][0]:.3f}   intercept a = {lr_all.intercept_[0]:+.3f}\n")

ev = df[mask]
for col in ['xg', 'xg_final']:
    print(f"EV 2023 onward using {col}:")
    print(bucket_table(ev.assign(dz=danger(ev[col])), col, 'dz'), '\n')

In [ ]:
# Final summary: calibration by season, and accuracy by state
print(bucket_table(df, 'xg_final', 'season')[['goals', 'xG', 'ratio']], '\n')
summary = pd.DataFrame({s: {'shots': len(g), 'goals': int(g.is_goal.sum()),
                            'xG/goals': round(g.xg_final.sum() / g.is_goal.sum(), 3),
                            'AUC': round(roc_auc_score(g.is_goal, g.xg_final), 3)}
                        for s, g in df.groupby('state', observed=True)}).T.astype({'shots': int, 'goals': int})
print(summary)
print(f"\nall situations: logloss {log_loss(df.is_goal, df.xg_final):.4f}  "
      f"AUC {roc_auc_score(df.is_goal, df.xg_final):.3f}")

## 17. Flurry adjustment and saving

A shot and its rebounds can add up to more than one expected goal on a single sequence, although at most one goal can come of it. The flurry adjustment multiplies each shot's xG by the chance that no earlier shot in the same flurry scored. A flurry is a run of shots by one team in one period, each within `FLURRY_GAP` seconds of the one before.

Use `xg_final` to judge a single shot's quality and to check calibration. Use `xg_flurry` when adding up xG for teams, players and goalies. Flurry-adjusted totals come out a little below actual goals. That is expected: `xg_final` is calibrated to match goals, and the adjustment only removes value.

Saves `xg_final.csv` to Drive with both columns, keyed by `game_id` and `event_id`.

In [ ]:
d = df.sort_values(['game_id', 'period', 'game_seconds'], kind='stable')
new = ((d.game_id != d.game_id.shift()) | (d.period != d.period.shift()) |
       (d.team != d.team.shift()) | (d.game_seconds - d.game_seconds.shift() > FLURRY_GAP))
fid = new.cumsum()
p = d.xg_final.clip(upper=0.999)
none_before = (1 - p).groupby(fid).cumprod() / (1 - p)      # chance no earlier shot in the flurry scored
df['xg_flurry'] = (p * none_before).reindex(df.index)
df['in_flurry'] = (fid.groupby(fid).transform('size') > 1).reindex(df.index)

t = df.groupby('season').agg(goals=('is_goal', 'sum'), xG=('xg_final', 'sum'), xG_flurry=('xg_flurry', 'sum'),
                             shots_in_flurries=('in_flurry', 'mean'))
t['flurry_vs_goals'] = (t.xG_flurry / t.goals).round(3)
print(t.round({'xG': 0, 'xG_flurry': 0, 'shots_in_flurries': 3}))
biggest = df.groupby(fid.reindex(df.index)).agg(shots=('xg_final', 'size'), xG=('xg_final', 'sum'),
                                               xG_flurry=('xg_flurry', 'sum')).nlargest(5, 'xG').round(2)
print("\nlargest flurries (raw xG can pass 1.0; adjusted cannot):")
print(biggest.to_string(index=False))

df[['game_id', 'event_id', 'xg_final', 'xg_flurry']].to_csv(f'{DATA_DIR}/xg_final.csv', index=False)
print("\nsaved", f'{DATA_DIR}/xg_final.csv')

## 18. xG by rink location

Average `xg_final` per shot at each spot in the attacking zone, on one shared color scale (red = most dangerous, blue = least). Empty-net shots are excluded. The image is also saved to Drive.

In [ ]:
from matplotlib.patches import Arc, Circle, Rectangle, PathPatch
from matplotlib.colors import PowerNorm
from matplotlib.path import Path

def rink_outline():
    """Closed path of the offensive-zone boards (rounded corners, radius 28 ft)."""
    t1 = np.linspace(np.pi, np.pi / 2, 30)
    t2 = np.linspace(np.pi / 2, 0, 30)
    pts = ([(-42.5, 25)] + [(-14.5 + 28 * np.cos(t), 72 + 28 * np.sin(t)) for t in t1] +
           [(14.5 + 28 * np.cos(t), 72 + 28 * np.sin(t)) for t in t2] + [(42.5, 25), (-42.5, 25)])
    return Path(pts, closed=True)

def draw_rink(ax):
    """Offensive zone, net at the top. Horizontal axis = y, vertical axis = x (feet)."""
    k, red, blue = dict(color='black', lw=1.2, zorder=3), '#c8102e', '#1f4e9e'
    ax.plot([-42.5, -42.5], [25, 72], **k); ax.plot([42.5, 42.5], [25, 72], **k)       # boards
    ax.add_patch(Arc((14.5, 72), 56, 56, theta1=0, theta2=90, **k))                    # corners
    ax.add_patch(Arc((-14.5, 72), 56, 56, theta1=90, theta2=180, **k))
    ax.plot([-14.5, 14.5], [100, 100], **k)                                            # end boards
    ax.plot([-42.5, 42.5], [25, 25], color=blue, lw=4, zorder=3)                       # blue line
    ax.plot([-36.75, 36.75], [89, 89], color=red, lw=1.5, zorder=3)                    # goal line
    ax.add_patch(Arc((0, 89), 12, 12, theta1=180, theta2=360, color=red, lw=1.5, zorder=3))  # crease
    ax.add_patch(Rectangle((-3, 89), 6, 3.3, fill=False, **k))                         # net
    ax.plot([-11, -14], [89, 100], **k); ax.plot([11, 14], [89, 100], **k)             # trapezoid
    for cy in (-22, 22):                                                               # faceoff circles
        ax.add_patch(Circle((cy, 69), 15, fill=False, color=red, lw=1.2, zorder=3))
        ax.plot(cy, 69, 'o', color=red, ms=4, zorder=3)
    ax.set_xlim(-44, 44); ax.set_ylim(24, 101.5); ax.set_aspect('equal'); ax.axis('off')

panels = [('EV', 'Even strength'), ('PP', 'Power play'), ('SH', 'Shorthanded')]
grid = {'EV': (30, 25), 'PP': (24, 15), 'SH': (18, 8)}   # (hexes across, min shots per hex)
oz = df[(df.x >= 25) & (df.x <= 100)]
norm = PowerNorm(gamma=0.5, vmin=0, vmax=0.30)           # shared scale; stretches the low end

fig, axes = plt.subplots(1, 3, figsize=(18, 6.8))
for ax, (s, name) in zip(axes, panels):
    d = oz[oz.state == s]
    gs, mc = grid[s]
    hb = ax.hexbin(d.y, d.x, C=d.xg_final, reduce_C_function=np.mean, gridsize=gs,
                   extent=(-42.5, 42.5, 25, 100), mincnt=mc, cmap='coolwarm', norm=norm,
                   linewidths=0.2, edgecolors='white')
    hb.set_clip_path(PathPatch(rink_outline(), transform=ax.transData))   # keep inside the boards
    draw_rink(ax)
    ax.set_title(f"{name}\n{len(d):,} shots · {int(d.is_goal.sum()):,} goals · "
                 f"avg xG {d.xg_final.mean():.3f}", fontsize=13)

cb = fig.colorbar(hb, ax=axes, shrink=0.75, pad=0.02)
cb.set_label('Average xG per shot', fontsize=12)
cb.set_ticks([0, .02, .05, .10, .15, .20, .30])
cb.set_ticklabels(['0%', '2%', '5%', '10%', '15%', '20%', '30%+'])
fig.suptitle(f'Expected goals by shot location, {df.season.min()}–{df.season.max()}', fontsize=16, y=1.0)
plt.savefig(f'{DATA_DIR}/xg_by_location.png', dpi=200, bbox_inches='tight')
plt.show()

## 19. What drives the model (SHAP)

A SHAP beeswarm for one of the four strength-state models. Each dot is one shot. How far right or left it sits is how much that feature raised or lowered that shot's xG (in log-odds: right = more dangerous). The colour is the feature's value on that shot, red for high and blue for low; features that are categories, such as shot type, are grey. Features are listed from most to least influential.

Every label gives the feature in plain words, then in square brackets the column of `shots_YYYY.csv` it comes from. Features built in section 4 show the columns they are built from. The same list is printed underneath as a table with each feature's share of the total influence, and the chart is saved to Drive as `xg_shap_EV.png`.

`SHAP_STATE` chooses the model: `'EV'`, `'PP'`, `'SH'` or `'EN'`.

**You do not need to re-run the whole notebook for this.** After a full run the cell explains the final model (every season, tuned settings). In a fresh session, run sections 1–5 and then this section: it fits one model with the starting settings, which takes a few minutes, and the picture is close to the tuned model's.

In [ ]:
!pip -q install shap

In [ ]:
import shap

SHAP_STATE = 'EV'      # which model to explain: 'EV' (even strength), 'PP', 'SH' or 'EN' (empty net)
SHAP_SHOTS = 20000     # shots sampled for the chart
SHAP_SHOW = 25         # how many features to show, most important first

# model feature: (plain description, column or columns in shots_YYYY.csv that it comes from)
LABELS = {
    'distance': ('Shot distance from the net', 'distance'),
    'angle': ('Shot angle from the middle of the net', 'angle'),
    'x': ('How far up the ice the shot was taken', 'x'),
    'y': ('Side-to-side position of the shot', 'y'),
    'shot_type': ('Shot type (wrist, slap, tip and so on)', 'shot_type'),
    'prev_event': ('What happened just before the shot', 'prev_event'),
    'prev_team_same': ('Previous event was by the shooting team', 'prev_team_same'),
    'prev_short_miss': ('Previous event was a shot that fell short', 'prev_short_miss'),
    'sec_since_prev': ('Seconds since the previous event', 'sec_since_prev'),
    'dist_from_prev': ('Distance from the previous event', 'dist_from_prev'),
    'speed': ('Puck speed since the previous event', 'dist_from_prev, sec_since_prev'),
    'rebound': ('Rebound', 'rebound'),
    'royal_road': ('Puck crossed the middle of the ice just before the shot', 'y, prev_y, sec_since_prev'),
    'behind_net': ('Shot from behind the goal line', 'x'),
    'strength': ('Skaters on the ice (5v5, 5v4 and so on)', 'strength'),
    'state': ('Strength state (even, power play, shorthanded, empty net)', 'strength, empty_net_against'),
    'empty_net_against': ('Shooting at an empty net', 'empty_net_against'),
    'score_diff': ('Score difference for the shooting team', 'score_diff'),
    'is_home': ('Shooter is on the home team', 'is_home'),
    'period': ('Period', 'period'),
    'game_seconds': ('Time elapsed in the game', 'game_seconds'),
    'is_D': ('Shooter is a defenseman', 'shooter_pos'),
    'prev_x': ('How far up the ice the previous event was', 'prev_x'),
    'prev_dist_net': ('Distance of the previous event from the net', 'prev_x, prev_y'),
    'dist_change': ('How far the puck moved toward the net', 'distance, prev_x, prev_y'),
    'angle_change': ('Change of angle since the previous event', 'x, y, prev_x, prev_y'),
    'angle_speed': ('Speed of that change of angle', 'x, y, prev_x, prev_y, sec_since_prev'),
    'rush': ('Rush shot (puck outside the zone within the last 4 seconds)', 'prev_x, sec_since_prev'),
    'sec_since_faceoff': ('Seconds since the last faceoff', 'sec_since_faceoff'),
    'sec_since_state_change': ('Seconds since the strength last changed', 'sec_since_state_change'),
    'shooter_toi': ('How long the shooter has been on the ice', 'shooter_toi'),
    'own_avg_toi': ("Average shift length of the shooter's team", 'own_avg_toi'),
    'opp_min_toi': ('Shift length of the freshest defender', 'opp_min_toi'),
    'opp_avg_toi': ('Average shift length of the defending team', 'opp_avg_toi'),
    'opp_max_toi': ('Shift length of the most tired defender', 'opp_max_toi'),
    'hand_L': ('Shooter is left-handed', 'shooter_hand'),
    'y_hand': ("Side of the ice relative to the shooter's stick hand", 'y, shooter_hand'),
    'off_wing': ('Shooting from his off wing', 'y, shooter_hand'),
    'sec_in_zone': ('Seconds the puck has been in the zone', 'sec_in_zone'),
    'team_shots_10s': ("Shooting team's shot attempts in the last 10 seconds", 'team_shots_10s'),
    'team_shots_30s': ("Shooting team's shot attempts in the last 30 seconds", 'team_shots_30s'),
    'prev2_event': ('The event two back', 'prev2_event'),
    'prev2_team_same': ('Event two back was by the shooting team', 'prev2_team_same'),
    'sec_since_prev2': ('Seconds since the event two back', 'sec_since_prev2'),
    'prev2_x': ('How far up the ice the event two back was', 'prev2_x'),
    'prev2_dist_net': ('Distance of the event two back from the net', 'prev2_x, prev2_y'),
    'season_cat': ('Season', 'game_id'),
}

df['season_cat'] = df.season.astype('category')
d = df[df.state == SHAP_STATE]
if 'ntrees' in globals():
    # the final model: every season, with the settings and tree counts from section 8, as in the backfill (section 14)
    feats = features + ['season_cat']
    model = new_model(SHAP_STATE, n_estimators=ntrees[SHAP_STATE])
    model.fit(d[feats], d.is_goal)
    trees, which = ntrees[SHAP_STATE], "the final model: every season, settings from section 8"
else:
    # section 8 has not run in this session: starting settings, stopped early on the validation season
    feats = features
    tr, va = d[d.season <= TRAIN_END], d[d.season == VALID]
    model = new_model(SHAP_STATE, hp_default, early_stopping_rounds=100)
    model.fit(tr[feats], tr.is_goal, sample_weight=0.5 ** ((VALID - tr.season) / H),
              eval_set=[(va[feats], va.is_goal)], verbose=False)
    trees, which = model.best_iteration + 1, "starting settings (section 8 has not run in this session)"

sample = d.sample(min(SHAP_SHOTS, len(d)), random_state=0)
contrib = model.get_booster().predict(xgb.DMatrix(sample[feats], enable_categorical=True), pred_contribs=True,
                                      iteration_range=(0, trees))
values = contrib[:, :-1]                 # one SHAP value per shot and feature, in log-odds of a goal
names = [f"{LABELS.get(f, (f, f))[0]}   [{LABELS.get(f, (f, f))[1]}]" for f in feats]
shown = sample[feats].copy()
for c in shown.select_dtypes('category').columns:
    shown[c] = shown[c].astype(str)      # categories have no low-to-high order, so their dots are drawn grey

shap.summary_plot(values, shown, feature_names=names, max_display=SHAP_SHOW, show=False,
                  plot_size=(13, 0.36 * min(SHAP_SHOW, len(feats)) + 1.8))
ax = plt.gca()
ax.set_xlabel('SHAP value: effect on the log-odds of a goal')
ax.set_title(f"What drives xG: {SHAP_STATE} model, {len(sample):,} shots", loc='left', fontsize=13)
plt.savefig(f'{DATA_DIR}/xg_shap_{SHAP_STATE}.png', dpi=200, bbox_inches='tight')
plt.show()

rank = pd.DataFrame({'feature': [LABELS.get(f, (f, f))[0] for f in feats], 'in the model as': feats,
                     'csv column': [LABELS.get(f, (f, f))[1] for f in feats],
                     'mean |SHAP|': np.abs(values).mean(axis=0)}).sort_values('mean |SHAP|', ascending=False)
rank.insert(0, 'rank', range(1, len(rank) + 1))
rank['share'] = (rank['mean |SHAP|'] / rank['mean |SHAP|'].sum() * 100).round(1).astype(str) + '%'
print(f"{SHAP_STATE} model ({which}; {trees} trees, {len(d):,} shots)")
print(rank.round(4).to_string(index=False))